# Stage 2 — Tokenizer

**Goal:** fit a vocabulary to the corpus, verify it round-trips exactly, and pack
the whole corpus into a flat token array for training.

### The decision that matters most in this notebook

We train a **SentencePiece** BPE, not a `tokenizers`-library BPE. That looks like
an arbitrary library preference. It is not — it is the difference between stage 8
working and stage 8 failing.

`convert_hf_to_gguf.py` resolves a Llama model's vocabulary like this:

```python
try:
    self._set_vocab_sentencepiece()      # needs a file named tokenizer.model
except FileNotFoundError:
    try:
        self._set_vocab_llama_hf()
    except (FileNotFoundError, TypeError):
        self._set_vocab_gpt2()           # hashes the pre-tokenizer
```

That last path compares a hash of the pre-tokenizer against a **hardcoded
registry** of known models and raises `NotImplementedError: BPE pre-tokenizer was
not recognized` for anything it hasn't seen — which is every custom
`tokenizers` BPE ever trained
([#8649](https://github.com/ggml-org/llama.cpp/issues/8649),
[#9927](https://github.com/ggml-org/llama.cpp/issues/9927)).

Because `_set_vocab_sentencepiece()` is tried **first** and does no hash lookup,
shipping a `tokenizer.model` file sidesteps the whole mechanism. One file, and
the landmine never arms.

### Why 8192 pieces

Vocabulary size is a model dimension, and at this scale the dominant one:

| vocab | embedding params | share of a ~15.7M model |
|---|---|---|
| 8,192 | 3.1M | 20% |
| 32,000 | 12.3M | 49% |
| 128,256 (Llama 3) | 49.2M | 76% — larger than the rest combined |

With tied embeddings the table is counted once, but the conclusion holds: reusing
a big off-the-shelf vocabulary would spend most of the parameter budget on tokens
this corpus never contains.

In [ ]:
# --- Colab bootstrap -------------------------------------------------------
# Set this to YOUR GitHub repo once; every notebook uses the same cell.
REPO_URL = "https://github.com/pythonstudentiam/e2e_llm_demo.git"

import os, subprocess, sys
from pathlib import Path

# transformers probes for TensorFlow and Flax and imports them if present.
# We never use either, and importing TF pulls in a protobuf dependency chain
# that breaks the moment anything reinstalls protobuf. Set before any
# transformers import.
os.environ["USE_TF"] = "0"
os.environ["USE_JAX"] = "0"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"

REPO = Path("/content/e2e_llm_demo")
WORK = Path("/content/work")          # scratch: data + checkpoints (ephemeral!)
WORK.mkdir(parents=True, exist_ok=True)

if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)

sys.path.insert(0, str(REPO / "src"))

# Colab ships torch; these are the rest. -q to keep the log readable.
%pip install -q sentencepiece "datasets>=3.0" "transformers>=4.45" "huggingface_hub>=0.30"

# HF token from the Colab Secrets panel (key icon, left sidebar). Name it
# HF_TOKEN and enable Notebook access -- the grant is PER NOTEBOOK, so every
# notebook asks separately. Never paste a token into a cell.
#
# login() rather than just setting the env var: it writes the token where every
# huggingface_hub call looks, including ones that ignore the environment.
try:
    from google.colab import userdata
    from huggingface_hub import login

    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
    print("HF_TOKEN loaded from Colab Secrets, authenticated")
except Exception as e:
    print("=" * 72)
    print(f"  HF_TOKEN IS NOT AVAILABLE  ({type(e).__name__}: {e})")
    print()
    print("  Every Hub call in this notebook will fail with 401 Unauthorized.")
    print("  Fix: click the key icon in the left sidebar, turn on Notebook")
    print("       access for HF_TOKEN, then RE-RUN THIS CELL before continuing.")
    print("=" * 72)

import torch
print(f"torch {torch.__version__} | CUDA {torch.cuda.is_available()} | "
      f"{torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")

In [ ]:
from tinyllm import config
from tinyllm.config import (
    model_cfg, train_cfg, data_cfg, tok_cfg, sft_cfg, gen_cfg, quant_cfg, serve_cfg, hub,
)

print(config.summary())

## 2.1 — Fit the vocabulary

Several non-default options here, each for a reason:

| option | why |
|---|---|
| `byte_fallback=True` | any byte is representable, so nothing is ever lossy. Costs 256 vocab slots |
| `normalization_rule_name="identity"` | disables NFKC so `decode(encode(s)) == s` exactly |
| `remove_extra_whitespaces=False` | same reason — whitespace survives round-tripping |
| `split_digits=True` | digits tokenize individually, so `1997` and `1998` aren't unrelated atoms |
| `user_defined_symbols` | ChatML control tokens stay atomic, never split into subwords |

Takes 2–4 minutes.

In [ ]:
from pathlib import Path
from tinyllm.tokenizer import train_sentencepiece, write_corpus_sample
from tinyllm.data import stream_texts

corpus_file = WORK / "tokenizer_corpus.txt"
if not corpus_file.exists():          # runtime was recycled since notebook 01
    print("regenerating the tokenizer corpus...")
    write_corpus_sample(stream_texts(split="train"), corpus_file)

tok_dir = WORK / "tokenizer"
sp_path = train_sentencepiece(corpus_file, tok_dir)
print(f"\ntrained -> {sp_path} ({sp_path.stat().st_size / 1e3:.0f} KB)")

## 2.2 — Inspect what BPE learned

The merges are readable, and reading them is the fastest way to understand what
BPE actually does: frequent character sequences get promoted into single tokens,
so common words become one token and rare words decompose into pieces.

In [ ]:
from tinyllm.tokenizer import load_sp, vocab_stats

sp = load_sp(sp_path)
stats = vocab_stats(sp)
for k, v in stats.items():
    print(f"  {k:<22} {v}")

print("\nA sample of learned pieces (ids 300-330):")
print("  " + "  ".join(repr(sp.IdToPiece(i)) for i in range(300, 330)))

print("\nHow specific words tokenize:")
for w in ["once", "upon", "Lily", "gloomy", "extraordinarily", "1997", "  spaced"]:
    ids = sp.EncodeAsIds(w)
    print(f"  {w!r:<20} -> {len(ids)} tokens: {[sp.IdToPiece(i) for i in ids]}")

Note `▁` (U+2581) marking a word boundary — SentencePiece encodes leading spaces
into the token itself, which is why it can round-trip whitespace exactly and why
` cat` and `cat` are different tokens.

## 2.3 — The stage 2 gate

Three assertions. Each catches a failure that is silent and expensive later:

1. **Round-trip exactness.** If the normalization settings regress to defaults,
   the corpus quietly changes underneath the model.
2. **Atomic control tokens.** If `<|im_start|>` splits into subwords, the chat
   template stops delimiting turns and stage 6 degrades for no visible reason.
3. **`tokenizer.model` exists.** The stage 8 landmine described above.

In [ ]:
from tinyllm.tokenizer import roundtrip_report, assert_special_tokens

# 1. round-trip on held-out text (not the text the vocabulary was fitted on)
held_out = list(stream_texts(split="validation", limit=1000))
rt = roundtrip_report(sp, held_out)

print(f"round-trip exact on {rt['n_texts']:,} held-out docs: {rt['exact']}")
if not rt["exact"]:
    for orig, back in rt["examples"]:
        print(f"  MISMATCH\n    in:  {orig[:100]!r}\n    out: {back[:100]!r}")
assert rt["exact"], "tokenizer is lossy -- check the normalization settings"

print(f"compression: {rt['chars_per_token']:.2f} chars/token "
      f"({rt['tokens_per_char']:.3f} tokens/char)")

# 2. control tokens are single ids
assert_special_tokens(sp)
print("special tokens atomic: OK")

In [ ]:
# 3. Build the HF wrapper and confirm tokenizer.model lands on disk.
from tinyllm.tokenizer import build_hf_tokenizer, assert_gguf_ready

hf_tok = build_hf_tokenizer(sp_path, tok_dir, chat_model=False)
assert_gguf_ready(tok_dir)

print("files in the tokenizer dir:")
for p in sorted(tok_dir.iterdir()):
    print(f"  {p.name:<28} {p.stat().st_size:>9,} B")
print("\ntokenizer.model present -> stage 8 will take the SentencePiece path.")

### Optional side quest: watch the landmine go off

You own this tokenizer, so this is the cheapest possible way to see the mechanism
that `tokenizer.model` is protecting you from. Train a `tokenizers`-library BPE,
save it *without* `tokenizer.model`, and try to convert it in stage 8 — you'll get
`NotImplementedError: BPE pre-tokenizer was not recognized`, and the fix is to
register your own hash in `get_vocab_base_pre()`.

Skip it if you'd rather keep moving. `docs/02-tokenizer.md` has the walkthrough.

## 2.4 — How many tokens per word, really?

Notebook 01 assumed ~1.3 tokens/word. Now we can check, and get the true token
budget for training.

In [ ]:
sample = held_out[:500]
n_words = sum(len(t.split()) for t in sample)
n_tokens = sum(len(sp.EncodeAsIds(t)) for t in sample)

print(f"  {n_tokens / n_words:.3f} tokens per word")
print(f"  {rt['chars_per_token']:.2f} characters per token")
print(f"\n  a {data_cfg.seq_len}-token context holds ~{data_cfg.seq_len / (n_tokens / n_words):.0f} words")

lens = [len(sp.EncodeAsIds(t)) + 2 for t in held_out]   # +2 for BOS/EOS
import numpy as np
print(f"  mean story length: {np.mean(lens):.0f} tokens")
print(f"  fraction fitting in {data_cfg.seq_len} tokens: {np.mean(np.array(lens) <= data_cfg.seq_len):.1%}")

## 2.5 — Pack the corpus

Now tokenize everything into a flat `uint16` array on disk. Three decisions:

- **`uint16`** — 8192 fits in 16 bits. As `int64` this would be 2.6 GB; it's 656 MB.
- **Flat, not per-document** — training samples random windows from one contiguous
  array. No padding, no ragged batches, zero wasted compute. Documents are
  separated by EOS, so windows that straddle a boundary teach the model where
  stories end. Without that it never learns to stop.
- **Validation from the dataset's own held-out split**, never a slice of train, so
  stage 5's perplexity can't be inflated by contamination.

This takes ~10–15 minutes for the full corpus. Set `SMOKE = True` to build a tiny
version in under a minute and come back for the real one later.

In [ ]:
from tinyllm.data import prepare_corpus

SMOKE = False   # True -> ~2k documents, for rehearsing the pipeline quickly

data_dir = WORK / "data"
stats = prepare_corpus(sp, data_dir, smoke=SMOKE)

print()
for split, s in stats.items():
    print(f"  {split:<6} {s['n_tokens']:>12,} tokens | {s['n_docs']:>9,} docs | {s['size_mb']:>7.1f} MB")

need = train_cfg.total_tokens
have = stats["train"]["n_tokens"]
print(f"\n  training budget needs {need:,} tokens; corpus has {have:,}")
print(f"  -> {need / have:.2f} epochs over the data")

Slightly over one epoch is a reasonable place to be. Much more than that and a
model this small starts memorizing rather than generalizing.

## 2.6 — Publish the tokenizer

The tokenizer is tiny (~500 KB) and *everything* downstream depends on it: token
ids are meaningless without it, so a checkpoint whose tokenizer was lost is
unrecoverable. Push it to the Hub now, before the runtime can be recycled.

In [ ]:
from huggingface_hub import HfApi

assert not hub.ckpt_repo.startswith("CHANGEME/"), (
    "Set HubConfig.user in src/tinyllm/config.py to your HF username, "
    "push to GitHub, then re-run the bootstrap cell."
)

api = HfApi()
api.create_repo(hub.ckpt_repo, repo_type="model", exist_ok=True, private=True)
for f in ["tokenizer.model", "tokenizer_config.json", "special_tokens_map.json"]:
    p = tok_dir / f
    if p.exists():
        api.upload_file(path_or_fileobj=str(p), path_in_repo=f"tokenizer/{f}",
                        repo_id=hub.ckpt_repo, repo_type="model")
        print(f"  pushed {f}")

print(f"\n-> https://huggingface.co/{hub.ckpt_repo}")

## 2.7 — Persist the packed corpus

`/content/work` dies with the runtime, and you are about to switch to a GPU
runtime, which kills this one. Repacking costs ~15 minutes — and it would cost it
*again* every time the 45-minute training run in stage 4 gets interrupted and
resumed.

The token files are ~660 MB, which is small enough to park on the Hub and pull
back in about a minute. Same reasoning as checkpointing: **anything expensive to
recompute should not live only on an ephemeral disk.**

In [ ]:
from huggingface_hub import HfApi

PUSH_DATA = True   # set False to skip and just repack in notebook 04

if PUSH_DATA:
    api = HfApi()
    api.create_repo(hub.ckpt_repo, repo_type="model", exist_ok=True, private=True)
    for name in ["train.bin", "train.json", "val.bin", "val.json"]:
        p = data_dir / name
        if not p.exists():
            continue
        print(f"uploading {name} ({p.stat().st_size / 1e6:.0f} MB)...")
        api.upload_file(path_or_fileobj=str(p), path_in_repo=f"data/{name}",
                        repo_id=hub.ckpt_repo, repo_type="model")
    print(f"\n-> https://huggingface.co/{hub.ckpt_repo}/tree/main/data")
    print("notebook 04 will pull these instead of repacking.")

## Stage 2 gate

- [x] Round-trip exact on 1,000 held-out documents
- [x] `<|im_start|>` / `<|im_end|>` are single atomic tokens
- [x] `tokenizer.model` on disk — stage 8 takes the SentencePiece path
- [x] `train.bin` and `val.bin` packed, validation uncontaminated
- [x] Tokenizer pushed to the Hub and safe from a runtime reset

**Next:** `03_architecture.ipynb`